In [ ]:
!pip install -q \
transformers==4.41.2 \
sentence-transformers==2.7.0 \
langchain==0.1.16 \
langchain-community==0.0.32 \
faiss-cpu \
pypdf

In [ ]:
from langchain.prompts import PromptTemplate

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain.chains import RetrievalQA

from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

In [ ]:
loader = PyPDFLoader("Datasheet_ESP32.PDF")
documents = loader.load()

print("Total pages:", len(documents))

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

docs = text_splitter.split_documents(documents)

print("Total chunks:", len(docs))

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings loaded ✅")

In [ ]:
vectorstore = FAISS.from_documents(docs, embeddings)

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text2text-generation",   # ✅ CORRECT
    model="google/flan-t5-large",
    max_length=256
)
from langchain_community.llms import HuggingFacePipeline

llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
prompt_template = """
Use the following context from a microcontroller datasheet to answer the question.
DO NOT copy the text directly.

IMPORTANT:
- Do NOT give short answers
- Always explain in detail.

If the answer is not present in the context, say:
"Not available in datasheet"

Context:
{context}

Question:
{question}

Answer:
"""
retriever = vectorstore.as_retriever()
PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": PROMPT}
)

In [ ]:
query = "What is SRAM specification? Explain"

raw_result = qa.run(query)

print(raw_result)

explanation = llm.invoke(
    f"Explain the {raw_result} "
)

print(explanation)